---
## 1. Data Dictionary

### Merged Dataset Variables

| Variable | Description | Data Type | Units | Possible Values / Notes |
|---|---|---|---|---|
| `entity` | Name of the country, territory, or aggregated region | Categorical (text) | N/A | e.g., "Afghanistan", "Brazil", "World", "High-income countries" |
| `code` | ISO 3166-1 alpha-3 country code | Categorical (text) | N/A | e.g., "AFG", "BRA"; **NaN for aggregated regions** (potential quality issue) |
| `year` | Year of observation | Numerical (integer) | Year | Range: 1751 to 2023 |
| `child_mortality_rate` | Under-5 mortality rate - probability of dying between birth and age 5 | Numerical (float) | Deaths per 100 live births | Range: ~0.3 to ~60+ |
| `fertility_rate_hist` | Total fertility rate - average number of children born per woman | Numerical (float) | Children per woman | Range: ~0.8 to ~9+ |

### Potential Data Quality Issues

- The `code` column contains **NaN values for aggregated entities** (e.g., "World", "High-income countries") that do not have ISO country codes.
- The two datasets have **different temporal coverages**: mortality data goes back to 1751 while fertility data starts in 1891, which will produce missing values after merging.
- The datasets have **different numbers of entities**: mortality has 213 entities while fertility has 261 entities. Not all entities appear in both datasets.
- Both columns `child_mortality_rate` and `fertility_rate_hist` are expected to be float but should be validated.

In [24]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [25]:
# Load both datasets
df_mortality = pd.read_csv('Dataset/child-mortality.csv')
df_fertility = pd.read_csv('Dataset/children-born-per-woman.csv')

print('=== Child Mortality Dataset ===')
print(f'Shape: {df_mortality.shape}')
print(df_mortality.head())
print()
print('=== Fertility Rate Dataset ===')
print(f'Shape: {df_fertility.shape}')
print(df_fertility.head())

=== Child Mortality Dataset ===
Shape: (16835, 4)
        entity code  year  child_mortality_rate
0  Afghanistan  AFG  1957                 37.13
1  Afghanistan  AFG  1958                 36.52
2  Afghanistan  AFG  1959                 35.95
3  Afghanistan  AFG  1960                 35.32
4  Afghanistan  AFG  1961                 34.76

=== Fertility Rate Dataset ===
Shape: (19402, 4)
        entity code  year  fertility_rate_hist
0  Afghanistan  AFG  1950                7.248
1  Afghanistan  AFG  1951                7.260
2  Afghanistan  AFG  1952                7.260
3  Afghanistan  AFG  1953                7.266
4  Afghanistan  AFG  1954                7.254


In [26]:
print('=== Child Mortality - Info ===')
print(df_mortality.info())
print()
print('=== Fertility Rate - Info ===')
print(df_fertility.info())

=== Child Mortality - Info ===
<class 'pandas.DataFrame'>
RangeIndex: 16835 entries, 0 to 16834
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                16835 non-null  str    
 1   code                  16835 non-null  str    
 2   year                  16835 non-null  int64  
 3   child_mortality_rate  16835 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 526.2 KB
None

=== Fertility Rate - Info ===
<class 'pandas.DataFrame'>
RangeIndex: 19402 entries, 0 to 19401
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   entity               19402 non-null  str    
 1   code                 19015 non-null  str    
 2   year                 19402 non-null  int64  
 3   fertility_rate_hist  19402 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 606.4 KB
None


In [27]:
print('=== Child Mortality - Descriptive Statistics ===')
print(df_mortality.describe())
print()
print('=== Fertility Rate - Descriptive Statistics ===')
print(df_fertility.describe())

=== Child Mortality - Descriptive Statistics ===
               year  child_mortality_rate
count  16835.000000          16835.000000
mean    1974.431838             10.726533
std       41.301284             10.787012
min     1751.000000              0.140000
25%     1960.000000              2.160000
50%     1984.000000              6.580000
75%     2004.000000             16.705000
max     2023.000000             76.740000

=== Fertility Rate - Descriptive Statistics ===
               year  fertility_rate_hist
count  19402.000000         19402.000000
mean    1985.861612             3.936123
std       22.049134             1.991179
min     1891.000000             0.662000
25%     1967.000000             2.138210
50%     1986.000000             3.466000
75%     2005.000000             5.849000
max     2023.000000             8.864000


In [28]:
# Merge datasets using outer join on the shared keys
df = pd.merge(df_mortality, df_fertility, on=['entity', 'code', 'year'], how='outer')

print(f'Merged dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

Merged dataset shape: (22260, 5)
Columns: ['entity', 'code', 'year', 'child_mortality_rate', 'fertility_rate_hist']



,entity,code,year,child_mortality_rate,fertility_rate_hist
0,Afghanistan,AFG,1950,NaN,7.248
1,Afghanistan,AFG,1951,NaN,7.260
2,Afghanistan,AFG,1952,NaN,7.260
3,Afghanistan,AFG,1953,NaN,7.266
4,Afghanistan,AFG,1954,NaN,7.254
5,Afghanistan,AFG,1955,NaN,7.262
6,Afghanistan,AFG,1956,NaN,7.269
7,Afghanistan,AFG,1957,37.13,7.264
8,Afghanistan,AFG,1958,36.52,7.269
9,Afghanistan,AFG,1959,35.95,7.276


In [29]:
print('=== Merged Dataset - Info ===')
df.info()
print()
print('=== Merged Dataset - Descriptive Statistics ===')
df.describe(include='all')

=== Merged Dataset - Info ===
<class 'pandas.DataFrame'>
RangeIndex: 22260 entries, 0 to 22259
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                22260 non-null  str    
 1   code                  21873 non-null  str    
 2   year                  22260 non-null  int64  
 3   child_mortality_rate  16835 non-null  float64
 4   fertility_rate_hist   19402 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 869.7 KB

=== Merged Dataset - Descriptive Statistics ===


,entity,code,year,child_mortality_rate,fertility_rate_hist
count,22260,21873,22260.000000,16835.000000,19402.000000
unique,262,255,NaN,NaN,NaN
top,Sweden,SWE,NaN,NaN,NaN
freq,273,273,NaN,NaN,NaN
mean,NaN,NaN,1975.151123,10.726533,3.936123
std,NaN,NaN,37.517296,10.787012,1.991179
min,NaN,NaN,1751.000000,0.140000,0.662000
25%,NaN,NaN,1959.000000,2.160000,2.138210
50%,NaN,NaN,1981.000000,6.580000,3.466000
75%,NaN,NaN,2002.000000,16.705000,5.849000


In [30]:
# 3.1.1 Check for exact (full-row) duplicates
exact_duplicates = df.duplicated().sum()
print(f'Number of exact (full-row) duplicates: {exact_duplicates}')

if exact_duplicates > 0:
    print('\nDuplicate rows:')
    print(df[df.duplicated(keep=False)])

Number of exact (full-row) duplicates: 0


In [31]:
# 3.1.2 Check for partial duplicates (same entity + code + year but different values)
key_cols = ['entity', 'code', 'year']
partial_duplicates = df.duplicated(subset=key_cols).sum()
print(f'Number of partial duplicates (same entity/code/year): {partial_duplicates}')

if partial_duplicates > 0:
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    print(f'\nShowing first conflicting entries:')
    print(df[dup_mask].sort_values(key_cols).head(20))

Number of partial duplicates (same entity/code/year): 0


In [32]:
# 3.1.3 Remove exact duplicates (keeping the first occurrence)
shape_before = df.shape[0]
df = df.drop_duplicates()
shape_after = df.shape[0]

print(f'Rows before duplicate removal: {shape_before}')
print(f'Rows after duplicate removal:  {shape_after}')
print(f'Rows removed: {shape_before - shape_after}')

Rows before duplicate removal: 22260
Rows after duplicate removal:  22260
Rows removed: 0


In [33]:
# Quantify missing values per variable
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct,
    'Non-Null Count': df.notnull().sum(),
    'Dtype': df.dtypes
})

print('=== Missing Values per Variable ===')
print(missing_summary)
print(f'\nTotal cells in dataset: {df.shape[0] * df.shape[1]}')
print(f'Total missing cells: {df.isnull().sum().sum()}')
print(f'Overall missing rate: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%')

=== Missing Values per Variable ===
                      Missing Count  Missing %  Non-Null Count    Dtype
entity                            0       0.00           22260      str
code                            387       1.74           21873      str
year                              0       0.00           22260    int64
child_mortality_rate           5425      24.37           16835  float64
fertility_rate_hist            2858      12.84           19402  float64

Total cells in dataset: 111300
Total missing cells: 8670
Overall missing rate: 7.79%


In [34]:
# Analyze where missing values occur
print('=== Rows with missing child_mortality_rate ===')
mort_missing = df[df['child_mortality_rate'].isnull()]
print(f'Count: {len(mort_missing)}')
print(f'Year range: {mort_missing["year"].min()} - {mort_missing["year"].max()}')
print(f'Sample entities: {mort_missing["entity"].unique()[:10]}')
print()

print('=== Rows with missing fertility_rate_hist ===')
fert_missing = df[df['fertility_rate_hist'].isnull()]
print(f'Count: {len(fert_missing)}')
print(f'Year range: {fert_missing["year"].min()} - {fert_missing["year"].max()}')
print(f'Sample entities: {fert_missing["entity"].unique()[:10]}')
print()

print('=== Rows with missing code ===')
code_missing = df[df['code'].isnull()]
print(f'Count: {len(code_missing)}')
print(f'Entities without code: {sorted(code_missing["entity"].unique()[:15])}')

df_without_interpolation = df

=== Rows with missing child_mortality_rate ===
Count: 5425
Year range: 1938 - 2023
Sample entities: <StringArray>
[   'Afghanistan',         'Africa',    'Africa (UN)',        'Albania',
        'Algeria', 'American Samoa',        'Andorra',         'Angola',
       'Anguilla',      'Argentina']
Length: 10, dtype: str

=== Rows with missing fertility_rate_hist ===
Count: 2858
Year range: 1751 - 2023
Sample entities: <StringArray>
['Antigua and Barbuda',           'Argentina',           'Australia',
             'Austria',            'Barbados',             'Belgium',
               'Benin',              'Brazil',              'Brunei',
        'Burkina Faso']
Length: 10, dtype: str

=== Rows with missing code ===
Count: 387
Entities without code: ['England and Wales', 'Least developed countries', 'Less developed regions', 'Less developed regions, excluding China', 'Less developed regions, excluding least developed countries', 'More developed regions', 'Scotland']


In [35]:
# Step 1: Sort by entity and year for proper interpolation
df = df.sort_values(['entity', 'year']).reset_index(drop=True)

print(f'Missing BEFORE treatment:')
print(df.isnull().sum())
print()

Missing BEFORE treatment:
entity                     0
code                     387
year                       0
child_mortality_rate    5425
fertility_rate_hist     2858
dtype: int64



In [36]:
# Step 2: Interpolate within each entity group (linear interpolation)
numeric_cols = ['child_mortality_rate', 'fertility_rate_hist']

df[numeric_cols] = df.groupby('entity')[numeric_cols].transform(
    lambda group: group.interpolate(method='linear', limit_direction='both')
)

print(f'Missing AFTER interpolation:')
print(df.isnull().sum())
print()

Missing AFTER interpolation:
entity                     0
code                     387
year                       0
child_mortality_rate    3495
fertility_rate_hist       56
dtype: int64



In [37]:
# Step 3: Remove rows where BOTH numeric columns are still NaN
# (entities that exist in only one dataset with no overlapping years)
both_null = df['child_mortality_rate'].isnull() & df['fertility_rate_hist'].isnull()
print(f'Rows with both numeric values missing: {both_null.sum()}')

shape_before = df.shape[0]
df = df.dropna(subset=numeric_cols, how='all')
shape_after = df.shape[0]

print(f'Rows removed: {shape_before - shape_after}')
print(f'\nMissing AFTER removal of fully-null rows:')
print(df.isnull().sum())

Rows with both numeric values missing: 0
Rows removed: 0

Missing AFTER removal of fully-null rows:
entity                     0
code                     387
year                       0
child_mortality_rate    3495
fertility_rate_hist       56
dtype: int64


In [38]:
# Step 4: For remaining single-column NaN values, apply median imputation
# grouped by entity (for entities that exist in only one dataset but share years)
for col in numeric_cols:
    remaining_nulls = df[col].isnull().sum()
    if remaining_nulls > 0:
        print(f'{col}: {remaining_nulls} remaining NaN values')
        # Fill with the entity's own median, falling back to global median
        entity_medians = df.groupby('entity')[col].transform('median')
        df[col] = df[col].fillna(entity_medians)
        # If still NaN (entity has no values at all), use global median
        df[col] = df[col].fillna(df[col].median())

print(f'\n=== Final Missing Values ===')
print(df.isnull().sum())
print(f'\nFinal dataset shape: {df.shape}')

child_mortality_rate: 3495 remaining NaN values
fertility_rate_hist: 56 remaining NaN values

=== Final Missing Values ===
entity                    0
code                    387
year                      0
child_mortality_rate      0
fertility_rate_hist       0
dtype: int64

Final dataset shape: (22260, 5)


In [39]:
# Final validation
print('=== Final Dataset Info ===')
df.info()
print()
print('=== Final Descriptive Statistics ===')
print(df.describe())
print()
print(f'Number of unique entities: {df["entity"].nunique()}')
print(f'Year range: {df["year"].min()} - {df["year"].max()}')
print(f'Total rows: {len(df)}')
print(f'Any remaining NaN (excl. code): {df.drop(columns="code").isnull().any().any()}')

=== Final Dataset Info ===
<class 'pandas.DataFrame'>
RangeIndex: 22260 entries, 0 to 22259
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                22260 non-null  str    
 1   code                  21873 non-null  str    
 2   year                  22260 non-null  int64  
 3   child_mortality_rate  22260 non-null  float64
 4   fertility_rate_hist   22260 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 869.7 KB

=== Final Descriptive Statistics ===
               year  child_mortality_rate  fertility_rate_hist
count  22260.000000          22260.000000         22260.000000
mean    1975.151123             10.578730             3.898180
std       37.517296             10.093923             1.918213
min     1751.000000              0.140000             0.662000
25%     1959.000000              2.870000             2.224000
50%     1981.000000              7.303750  

In [40]:
# Preview the cleaned dataset
print('=== First 10 rows of the cleaned dataset ===')
df.head(10)

=== First 10 rows of the cleaned dataset ===


,entity,code,year,child_mortality_rate,fertility_rate_hist
0,Afghanistan,AFG,1950,37.13,7.248
1,Afghanistan,AFG,1951,37.13,7.260
2,Afghanistan,AFG,1952,37.13,7.260
3,Afghanistan,AFG,1953,37.13,7.266
4,Afghanistan,AFG,1954,37.13,7.254
5,Afghanistan,AFG,1955,37.13,7.262
6,Afghanistan,AFG,1956,37.13,7.269
7,Afghanistan,AFG,1957,37.13,7.264
8,Afghanistan,AFG,1958,36.52,7.269
9,Afghanistan,AFG,1959,35.95,7.276


In [41]:
df.to_csv('Dataset/merged_cleaned_dataset.csv', index=False)
print('Cleaned dataset saved to: Dataset/merged_cleaned_dataset.csv')
print(f'Final shape: {df.shape}')

Cleaned dataset saved to: Dataset/merged_cleaned_dataset.csv
Final shape: (22260, 5)


In [56]:
print("DataFrame - Without Interpolation")
print(df_without_interpolation.describe())
print(df.shape)

DataFrame - Without Interpolation
               year  child_mortality_rate  fertility_rate_hist
count  22260.000000          16835.000000         19402.000000
mean    1975.151123             10.726533             3.936123
std       37.517296             10.787012             1.991179
min     1751.000000              0.140000             0.662000
25%     1959.000000              2.160000             2.138210
50%     1981.000000              6.580000             3.466000
75%     2002.000000             16.705000             5.849000
max     2023.000000             76.740000             8.864000
(22260, 5)


In [57]:
print("DataFrame - With Interpolation")
print(df.describe())
print(df.shape)
#separar por regiões de forma mais correta para depois colocar em mapa
#se tiver agrupamento por um e não por outro, descobrir e fazer o agrupamento corretamente no que não o agrupamento
#agrupamento por periodo de tempo, por regiões
#separar por paises e fazer uns graficos

DataFrame - With Interpolation
               year  child_mortality_rate  fertility_rate_hist
count  22260.000000          22260.000000         22260.000000
mean    1975.151123             10.578730             3.898180
std       37.517296             10.093923             1.918213
min     1751.000000              0.140000             0.662000
25%     1959.000000              2.870000             2.224000
50%     1981.000000              7.303750             3.429500
75%     2002.000000             14.982500             5.719500
max     2023.000000             76.740000             8.864000
(22260, 5)


<StringArray>
[        'Afghanistan',              'Africa',         'Africa (UN)',
             'Albania',             'Algeria',      'American Samoa',
             'Andorra',              'Angola',            'Anguilla',
 'Antigua and Barbuda',
 ...
             'Vanuatu',             'Vatican',           'Venezuela',
             'Vietnam',   'Wallis and Futuna',      'Western Sahara',
               'World',               'Yemen',              'Zambia',
            'Zimbabwe']
Length: 262, dtype: str

In [54]:
df_original = df_without_interpolation.sort_values(['entity', 'year']).reset_index(drop=True)

df_interpolated_before = df_original[mask_interpolated].copy()
df_interpolated_before['mortality_interpolated'] = mask_mortality_null[mask_interpolated].values
df_interpolated_before['fertility_interpolated'] = mask_fertility_null[mask_interpolated].values

print('DataFrame - Interpolated Rows')
print(df_interpolated_before.describe())
print(df_interpolated_before.shape)

DataFrame - Interpolated Rows
              year  child_mortality_rate  fertility_rate_hist
count  8283.000000           2858.000000          5425.000000
mean   1951.524931             24.018286             3.957222
std      46.171734             11.535558             1.972207
min    1751.000000              0.370000             0.662000
25%    1931.000000             14.930000             2.215000
50%    1958.000000             24.085000             3.484000
75%    1981.000000             31.547500             5.888000
max    2023.000000             68.210000             8.198000
(8283, 7)
